# RAG Pipeline Experiments

Learn the pipeline stage by stage: LOAD → CHUNK (sections) → EMBED → STORE → RETRIEVE → ANSWER.

Select the `backend/.venv` kernel. Requires `GEMINI_API_KEY` in `backend/.env`.

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

backend_dir = Path.cwd().parent          # .../backend
sys.path.insert(0, str(backend_dir))
load_dotenv(backend_dir / ".env")

import os
assert os.getenv("GEMINI_API_KEY"), "Add GEMINI_API_KEY to backend/.env"
print("setup OK —", backend_dir)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/Users/saiteja/Library/Mobile Documents/com~apple~CloudDocs/Downloads/SriVenkataSivaSaiTejaTankala_Resume.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"pages: {len(pages)}")
print(pages[0].page_content[:400])

In [ ]:
from app.services.document_processor import detect_sections

section_docs = detect_sections(pages)
print(f"section documents: {len(section_docs)}")
for doc in section_docs:
    print(f"  p.{doc.metadata['page']}  [{doc.metadata['section']}]  {len(doc.page_content)} chars")

print("\nsample section doc:\n", section_docs[0].page_content[:300])

In [ ]:
from langchain_community.embeddings import FastEmbedEmbeddings

embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector = embeddings.embed_query(section_docs[0].page_content[:500])
print(f"embedding dims: {len(vector)}")
print("first 5 values:", vector[:5])

In [ ]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(section_docs)
doc_name = Path(PDF_PATH).stem
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = i
    chunk.page_content = f"{doc_name} > {chunk.metadata['section']}\n\n{chunk.page_content}"
print(f"chunks: {len(chunks)}")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_experiment",
    persist_directory="./chroma_db",
)
print("stored count:", vectordb._collection.count())

In [ ]:
QUESTION = "What is Teja's most recent work experience?"

results = vectordb.similarity_search_with_score(QUESTION, k=3)
for doc, score in results:
    print(f"score={score:.3f}  p.{doc.metadata['page']}  [{doc.metadata['section']}]")
    print(doc.page_content[:200], "\n---")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

prompt = PromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GEMINI_API_KEY"))
    | StrOutputParser()
)

print(chain.invoke(QUESTION))

In [ ]:
# Experiment: change k and the question, see how retrieval and the answer change.
for k in (1, 3, 10):
    hits = vectordb.similarity_search(QUESTION, k=k)
    sections = [f"p.{h.metadata['page']}[{h.metadata['section']}]" for h in hits]
    print(f"k={k}: {sections}")